100ページ（見開きPNG）を一括で「右/左ページに分割 → 紙面トリム → 文字トリム（gutter側余白確保）」して保存する

出力パスはすべて ../../../output 前提

In [ ]:
# Environment setup
import os
import glob
import re
import cv2
import numpy as np

# ========= Paths =========
SPREAD_GLOB = "../../../output/purge_part2_1/spreads/spread_*.png"

OUT_RIGHT_DIR = "../../../output/purge_part2_1/pages_cropped/right"
OUT_LEFT_DIR  = "../../../output/purge_part2_1/pages_cropped/left"
OUT_DEBUG_DIR = "../../../output/purge_part2_1/pages_cropped/_debug"  # 任意（gutter値ログなど）

os.makedirs(OUT_RIGHT_DIR, exist_ok=True)
os.makedirs(OUT_LEFT_DIR, exist_ok=True)
os.makedirs(OUT_DEBUG_DIR, exist_ok=True)

In [ ]:
# ========= Tunables (you can adjust if needed) =========
GUTTER_CUT = 30       # ノド中心から左右に捨てるpx
WHITE_THRESH = 200    # 紙面検出：白っぽさの閾値（190〜215で調整）
PAPER_PAD = 6         # 紙面トリムの余白
INK_THRESH = 165      # 文字検出：インク（暗い）閾値（155〜175で調整）
EDGE_IGNORE_TEXT = 0.06  # 文字トリム時に外周を無視する割合
EDGE_IGNORE_PAPER = 0.02 # 紙面トリム時に外周を少し無視

# gutter側だけ余白を厚くする（切りすぎ防止）
GUTTER_PAD = 45
OTHER_PAD = 12
PAD_TOP = 10
PAD_BOTTOM = 10

# ========= Helpers =========
def detect_gutter_x(bgr: np.ndarray) -> int:
    """中央付近(45%〜55%)の縦方向平均からノド位置を推定（暗い=影が強い所をgutterと推定）"""
    h, w = bgr.shape[:2]
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    top = int(h * 0.10)
    bottom = int(h * 0.95)
    vmean = gray[top:bottom, :].mean(axis=0)

    L = int(w * 0.45)
    R = int(w * 0.55)
    gx = L + int(np.argmin(vmean[L:R]))
    return gx


def split_spread(bgr: np.ndarray, gutter_cut: int = 30):
    """
    見開きから物理右ページ/物理左ページを返す（読み順を意識して right -> left の順）
    物理左ページ = x < gutter
    物理右ページ = x > gutter
    """
    gx = detect_gutter_x(bgr)
    left = bgr[:, :max(0, gx - gutter_cut)]
    right = bgr[:, min(bgr.shape[1], gx + gutter_cut):]
    return right, left, gx


def crop_to_paper_strict(bgr: np.ndarray, pad: int = 6, white_thresh: int = 200, edge_ignore: float = 0.02):
    """
    明るい領域(=紙面)を優先して最大成分を切り出す。
    台/背景が薄くても、中心にある大きい白領域を拾う。
    """
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    ex = int(w * edge_ignore)
    ey = int(h * edge_ignore)
    inner = gray[ey:h - ey, ex:w - ex]

    mask = (inner >= white_thresh).astype(np.uint8) * 255

    k = cv2.getStructuringElement(cv2.MORPH_RECT, (35, 35))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=2)

    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return bgr

    cx0, cy0 = (mask.shape[1] / 2), (mask.shape[0] / 2)

    def score(c):
        area = cv2.contourArea(c)
        M = cv2.moments(c)
        if M["m00"] == 0:
            return -1
        cx = M["m10"] / M["m00"]
        cy = M["m01"] / M["m00"]
        dist = (cx - cx0) ** 2 + (cy - cy0) ** 2
        return area - 0.001 * dist

    c = max(cnts, key=score)
    x, y, ww, hh = cv2.boundingRect(c)

    x += ex
    y += ey

    x1 = max(0, x - pad)
    y1 = max(0, y - pad)
    x2 = min(w, x + ww + pad)
    y2 = min(h, y + hh + pad)
    return bgr[y1:y2, x1:x2]


def crop_to_text_asym_strict(page_bgr: np.ndarray,
                            ink_thresh: int = 165,
                            pad_top: int = 10,
                            pad_bottom: int = 10,
                            pad_left: int = 10,
                            pad_right: int = 10,
                            edge_ignore: float = 0.06):
    """
    文字(インク)領域で詰めるが、外周 edge_ignore% は無視して
    台の縁/黒枠/影を拾わないようにする。
    """
    gray = cv2.cvtColor(page_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    ex = int(w * edge_ignore)
    ey = int(h * edge_ignore)
    inner = gray[ey:h - ey, ex:w - ex]

    mask = (inner < ink_thresh).astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)

    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return page_bgr

    x1 = xs.min() + ex
    x2 = xs.max() + ex
    y1 = ys.min() + ey
    y2 = ys.max() + ey

    x1 = max(0, x1 - pad_left)
    x2 = min(w - 1, x2 + pad_right)
    y1 = max(0, y1 - pad_top)
    y2 = min(h - 1, y2 + pad_bottom)

    return page_bgr[y1:y2, x1:x2]


def extract_spread_index(path: str) -> int:
    """
    spread_002.png -> 2
    取れなければ 10**9 を返して末尾へ
    """
    m = re.search(r"spread_(\d+)", os.path.basename(path))
    if not m:
        return 10**9
    return int(m.group(1))


In [ ]:
# ========= Main loop =========
spread_paths = sorted(glob.glob(SPREAD_GLOB), key=extract_spread_index)

if not spread_paths:
    raise FileNotFoundError(f"No files matched: {SPREAD_GLOB}")

print(f"Found {len(spread_paths)} spreads")

log_path = os.path.join(OUT_DEBUG_DIR, "gutter_log.csv")
with open(log_path, "w", encoding="utf-8") as log:
    log.write("spread_file,gutter_x,spread_w,spread_h\n")

    for idx, sp in enumerate(spread_paths, start=1):
        img = cv2.imread(sp)
        if img is None:
            print(f"[WARN] unreadable: {sp}")
            continue

        h, w = img.shape[:2]

        # 1) split
        right_page, left_page, gx = split_spread(img, gutter_cut=GUTTER_CUT)

        # 2) paper crop (both)
        right_paper = crop_to_paper_strict(right_page, pad=PAPER_PAD, white_thresh=WHITE_THRESH, edge_ignore=EDGE_IGNORE_PAPER)
        left_paper  = crop_to_paper_strict(left_page,  pad=PAPER_PAD, white_thresh=WHITE_THRESH, edge_ignore=EDGE_IGNORE_PAPER)

        # 3) text crop (both) with gutter-side padding thicker
        # right page: gutter is LEFT
        right_crop = crop_to_text_asym_strict(
            right_paper,
            ink_thresh=INK_THRESH,
            pad_top=PAD_TOP,
            pad_bottom=PAD_BOTTOM,
            pad_left=GUTTER_PAD,
            pad_right=OTHER_PAD,
            edge_ignore=EDGE_IGNORE_TEXT
        )

        # left page: gutter is RIGHT
        left_crop = crop_to_text_asym_strict(
            left_paper,
            ink_thresh=INK_THRESH,
            pad_top=PAD_TOP,
            pad_bottom=PAD_BOTTOM,
            pad_left=OTHER_PAD,
            pad_right=GUTTER_PAD,
            edge_ignore=EDGE_IGNORE_TEXT
        )

        # 4) save
        base = os.path.splitext(os.path.basename(sp))[0]  # spread_002
        out_r = os.path.join(OUT_RIGHT_DIR, f"{base}_R.png")
        out_l = os.path.join(OUT_LEFT_DIR,  f"{base}_L.png")

        cv2.imwrite(out_r, right_crop)
        cv2.imwrite(out_l, left_crop)

        # 5) log gutter
        log.write(f"{os.path.basename(sp)},{gx},{w},{h}\n")

        if idx % 10 == 0:
            print(f"Processed {idx}/{len(spread_paths)}")

print("✅ Done.")
print("Outputs:")
print("  Right:", OUT_RIGHT_DIR)
print("  Left :", OUT_LEFT_DIR)
print("  Log  :", log_path)
